# E-Commerce Intelligence System

## 01 — Data Profiling

### Objective

Understand the structure, quality, and relationships of the raw Olist e-commerce datasets before performing data cleaning, transformation, and analysis.

### Dataset

Olist Brazilian E-Commerce Public Dataset

### Phase

Phase 1 — Data Profiling

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/raw")

In [3]:
DATA_DIR.exists()

True

In [4]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
tables = {}

for file in csv_files:
    table_name = file.stem
    tables[table_name] = pd.read_csv(file)
    

In [6]:
tables


{'olist_customers_dataset':                             customer_id                customer_unique_id  \
 0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 ...                                 ...                               ...   
 99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
 99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
 99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
 99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
 99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   
 
        customer_zip_code_prefix   

In [7]:
tables.keys()


dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [8]:
tables["olist_orders_dataset"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
tables.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [10]:
dataset_inventory = []

for table_name, df in tables.items():
    dataset_inventory.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_inventory = pd.DataFrame(dataset_inventory)

dataset_inventory

,table,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
4,olist_order_reviews_dataset,99224,7
5,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


## 2. Schema and Data Quality Profiling

We inspect each table's columns, data types, missing values, and cardinality.


In [11]:
def profile_table(df, table_name):
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean() * 100).round(2).values,
        "unique": df.nunique().values
    })

    print(f"Table: {table_name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")

    return profile


In [12]:
orders = tables["olist_orders_dataset"]

orders_profile = profile_table(
    orders,
    "olist_orders_dataset"
)

orders_profile

Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,unique
0,order_id,object,99441,0,0.00,99441
1,customer_id,object,99441,0,0.00,99441
2,order_status,object,99441,0,0.00,8
3,order_purchase_timestamp,object,99441,0,0.00,98875
4,order_approved_at,object,99281,160,0.16,90733
5,order_delivered_carrier_date,object,97658,1783,1.79,81018
6,order_delivered_customer_date,object,96476,2965,2.98,95664
7,order_estimated_delivery_date,object,99441,0,0.00,459


In [13]:
profiles = {}

for table_name, df in tables.items():
    profiles[table_name] = profile_table(df, table_name)

Table: olist_customers_dataset
Rows: 99,441
Columns: 5
Duplicate rows: 0
Table: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Duplicate rows: 261,831
Table: olist_order_items_dataset
Rows: 112,650
Columns: 7
Duplicate rows: 0
Table: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Duplicate rows: 0
Table: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Duplicate rows: 0
Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0
Table: olist_products_dataset
Rows: 32,951
Columns: 9
Duplicate rows: 0
Table: olist_sellers_dataset
Rows: 3,095
Columns: 4
Duplicate rows: 0
Table: product_category_name_translation
Rows: 71
Columns: 2
Duplicate rows: 0


In [14]:
for table_name, profile in profiles.items():
    print("\n" + "=" * 80)
    print(table_name)
    display(profile)


olist_customers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,customer_id,object,99441,0,0.0,99441
1,customer_unique_id,object,99441,0,0.0,96096
2,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,customer_city,object,99441,0,0.0,4119
4,customer_state,object,99441,0,0.0,27



olist_geolocation_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
1,geolocation_lat,float64,1000163,0,0.0,717360
2,geolocation_lng,float64,1000163,0,0.0,717613
3,geolocation_city,object,1000163,0,0.0,8011
4,geolocation_state,object,1000163,0,0.0,27



olist_order_items_dataset


...

,column,dtype,non_null,missing,missing_pct,unique
0,product_id,object,32951,0,0.00,32951
1,product_category_name,object,32341,610,1.85,73
2,product_name_lenght,float64,32341,610,1.85,66
3,product_description_lenght,float64,32341,610,1.85,2960
4,product_photos_qty,float64,32341,610,1.85,19
5,product_weight_g,float64,32949,2,0.01,2204
6,product_length_cm,float64,32949,2,0.01,99
7,product_height_cm,float64,32949,2,0.01,102
8,product_width_cm,float64,32949,2,0.01,95



olist_sellers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,seller_id,object,3095,0,0.0,3095
1,seller_zip_code_prefix,int64,3095,0,0.0,2246
2,seller_city,object,3095,0,0.0,611
3,seller_state,object,3095,0,0.0,23



product_category_name_translation


,column,dtype,non_null,missing,missing_pct,unique
0,product_category_name,object,71,0,0.0,71
1,product_category_name_english,object,71,0,0.0,71


In [15]:
profiles = {}

for table_name, df in tables.items():
    profiles[table_name] = profile_table(df, table_name)

Table: olist_customers_dataset
Rows: 99,441
Columns: 5
Duplicate rows: 0
Table: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Duplicate rows: 261,831
Table: olist_order_items_dataset
Rows: 112,650
Columns: 7
Duplicate rows: 0
Table: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Duplicate rows: 0
Table: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Duplicate rows: 0
Table: olist_orders_dataset
Rows: 99,441
Columns: 8
Duplicate rows: 0
Table: olist_products_dataset
Rows: 32,951
Columns: 9
Duplicate rows: 0
Table: olist_sellers_dataset
Rows: 3,095
Columns: 4
Duplicate rows: 0
Table: product_category_name_translation
Rows: 71
Columns: 2
Duplicate rows: 0


In [16]:

for table_name, profile in profiles.items():
    print("\n" + "=" * 80)
    print(table_name)
    display(profile)


olist_customers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,customer_id,object,99441,0,0.0,99441
1,customer_unique_id,object,99441,0,0.0,96096
2,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,customer_city,object,99441,0,0.0,4119
4,customer_state,object,99441,0,0.0,27



olist_geolocation_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
1,geolocation_lat,float64,1000163,0,0.0,717360
2,geolocation_lng,float64,1000163,0,0.0,717613
3,geolocation_city,object,1000163,0,0.0,8011
4,geolocation_state,object,1000163,0,0.0,27



olist_order_items_dataset


...

,column,dtype,non_null,missing,missing_pct,unique
0,product_id,object,32951,0,0.00,32951
1,product_category_name,object,32341,610,1.85,73
2,product_name_lenght,float64,32341,610,1.85,66
3,product_description_lenght,float64,32341,610,1.85,2960
4,product_photos_qty,float64,32341,610,1.85,19
5,product_weight_g,float64,32949,2,0.01,2204
6,product_length_cm,float64,32949,2,0.01,99
7,product_height_cm,float64,32949,2,0.01,102
8,product_width_cm,float64,32949,2,0.01,95



olist_sellers_dataset


,column,dtype,non_null,missing,missing_pct,unique
0,seller_id,object,3095,0,0.0,3095
1,seller_zip_code_prefix,int64,3095,0,0.0,2246
2,seller_city,object,3095,0,0.0,611
3,seller_state,object,3095,0,0.0,23



product_category_name_translation


,column,dtype,non_null,missing,missing_pct,unique
0,product_category_name,object,71,0,0.0,71
1,product_category_name_english,object,71,0,0.0,71
